# M06-01 — Explain y DAG

[← Anterior](01-teoria.ipynb) · [Siguiente →](03-lab-cache-particionado.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Dejar de pensar como Pandas: escribir `where` **no** recorre el fichero. Spark solo alarga una **receta**. `explain` es ver qué haría **sin** hacerlo. `count` es hacerlo (ahí nace un job en la UI).

Orden de este lab (si lo inviertes, no se nota nada): **print → explain → count**. Primero el mapa, luego el viaje.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M06-01-explain-dag.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


## Qué queremos conseguir (léelo; el código viene después)

Vienes de Pandas (o de “ejecuto la celda y ya está filtrado”). En Spark **no**.

Hasta que no lanzas una **acción** (`count`, `show`, `write`), Spark **no** ha leído las 1980 líneas de `fact_lines`. Los `where` / `select` solo alargan una receta. Por eso `print(planned)` no te da un número: te enseña el **objeto** (el papel de la receta).

`explain` es la herramienta de este lab: **ver qué haría Spark sin recorrerlo**. El muro de texto no se lee como un libro; se **busca** Scan (de dónde salen las filas) y Filter (tus `where`).

`count` es el viaje: ahora sí abre el Parquet, aplica la receta y te da un entero. En Spark UI aparece **un job nuevo**. El dibujo de ese job (etapas) y el árbol de la pestaña SQL son *la misma historia* que el `explain`, pero **después** de haber ejecutado.

Eso es todo. No memorices Catalyst. El lab siguiente (cache) solo tiene sentido si esto está claro: “esta receta es cara → la guardo”.

| Escribes | ¿Recorre `fact_lines`? | Qué ves en el notebook | Qué ves en la UI |
|----------|------------------------|------------------------|------------------|
| `where` / `select` / `read` | no | nada (o el objeto si haces `print`) | Jobs **igual** |
| `print(df)` | no | `DataFrame[columnas…]`, **sin** entero | Jobs **igual** |
| `df.explain(...)` | no | muro de texto (receta) | Jobs **igual** |
| `df.count()` / `show()` | **sí** | un entero / filas | **un job nuevo** |

El número de filas del `count` **no** es la demo (ya sabías filtrar en M03). La demo es: Job Id **quieto** con print/explain, Job Id **+1** con count.

### Markdown de *tu* notebook: no improvises teoría

El plantilla de los pasos dice “con tus palabras”. **Aquí no.** En cada paso te dejo el párrafo ya redactado, con huecos `___` (Job Ids, sí/no). Cópialo, ejecuta, rellena. Si escribes otra teoría, este lab vuelve a “no demostrar nada”.

Necesitas `data/staging/fact_lines` (M03-02 o `python3 scripts/run_pipeline.py`).


## Spark UI: ábrela **antes** del paso 1 (una sola vez)

Sin esto, cada paso te va a pedir “mira Jobs” y no vas a saber a qué ventana te refieres.

1. En el Codespace / VS Code, panel de abajo: pestaña **Ports** (junto a Terminal). Si no está: *View → Ports*.
2. Busca el puerto **4040**. Suele poner `Spark` o el nombre de la app.
3. Clic en el **globo** (Open in Browser) o *Forward*. Se abre una web “Spark … Application UI”.
4. Arriba, el nombre de la app debe ser `novashop-m06` **después** de ejecutar `get_spark` del paso 1. Si aún no lo has ejecutado, abre la UI igual y **recarga** tras el paso 1.
5. Si **4040** no carga o está vacío: mira si hay **4041**. Otra sesión se quedó el 4040. En una celda: `spark.stop()`, otra vez `get_spark("novashop-m06")`, recarga Ports y entra al puerto que haya salido.
6. Pestañas que usamos en **este** lab (las otras, ignóralas):

| Pestaña | Qué es, en cristiano | Cuándo hay algo que mirar |
|---------|----------------------|---------------------------|
| **Jobs** | Lista de **viajes**. Cada `count`/`show`/`write` = una fila nueva. | Solo **después** de una acción. Print y explain **no** añaden fila. |
| **SQL / DataFrame** | El **árbol** de esa consulta (Scan, Filter, Aggregate…). Es el `explain` después de haber viajado. | Después del primer `count`. |
| **Storage** | Datasets guardados en memoria (`cache`). | En este lab debe quedarse **vacío**. Si no, kernel sucio: `spark.catalog.clearCache()`. |

**Cómo se lee Jobs** (la tabla, no el dibujo):

- **Job Id** — 0, 1, 2… El más alto es el último viaje. Si ya corriste la teoría en este kernel, **no empieza en 0**. Da igual: lo que importa es si **sube** o no.
- **Description** — casi siempre se ve `count` (o `show`). Es el nombre de la acción, no de tu DataFrame.
- **Duration** — en local puede ser 0,1 s. No es la prueba.
- **Stages** — `2/2` o `1/1`. Un job se parte en **etapas** cuando hay shuffle. Un `count` típico: etapa que lee+filtra+cuenta trozos, a veces otra que junta el total (`Exchange` por medio).

**El dibujo (DAG) no es el `explain`.**

- En **Jobs**, al cliclar un Job Id, el “DAG Visualization” son **cajas grandes = stages**. Vas a ver nombres tipo *WholeStageCodegen*, *Exchange*, *count*. **No busques la palabra `Filter` ahí.** El filtro va *dentro* de un stage y esa caja no lo detalla.
- El árbol con *Scan parquet* y *Filter* está en **SQL / DataFrame** → clic en la consulta → Details. Eso **sí** se parece al `explain` del notebook.
- Si abres un DAG y el Job Id es **menor o igual** al que anotaste *antes* de la celda, es un viaje **viejo**. No es de esta celda.

**Recarga.** La UI no siempre pinta sola: vuelve a cliclar **Jobs** (o F5) después de cada celda.

Anota **ahora**, antes de pegar código, el Job Id más alto que ves (si no hay ninguno, escribe `ninguno / -1`):

`Job Id de partida = ___`


### Paso 1 — Encadenar no ejecuta (solo alarga la receta)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Ignora lo de “con tus palabras”. Copia esto en el Markdown y rellena **después** de ejecutar:

```
Paso 1. Escribir where no recorre el Parquet: solo alarga la receta.
print(planned) me enseñó el objeto DataFrame[…], no un entero.
Job Id de partida: ___
Job Id después de esta celda: ___   (tiene que ser el mismo)
En Jobs no hay fila nueva. Storage sigue vacío. No hay DAG nuevo que abrir.
```

**UI de este paso (hazlo en este orden):**

1. Mira Jobs. Anota el Job Id más alto (el de partida, si aún no lo tenías).
2. Ejecuta la celda de código (`Shift+Enter`).
3. Vuelve a la UI → **Jobs** (recarga). El más alto **no** cambia.
4. Abre **Storage**. Vacío.
5. **No** clicles un job viejo para “ver el DAG”. Ese DAG no es de esta celda. Esta celda **no** ha viajado.

Si el Job Id **sube** aquí: tienes un `count`/`show` de más en la celda, o `get_spark` disparó algo raro. Quita cualquier `.count()` y reejecuta.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `DataFrame[order_id, customer_id, gmv_line, order_month]`. Ningún entero tipo 800. Job Id **igual** que el de partida.

**Por qué este paso.** Si `print` ya te diera 1127, estarías ejecutando un `count` sin darte cuenta. Pandas ya habría filtrado; Spark no.

**Si no sale.** PATH / AnalysisException de `fact_lines` → falta M03-02 o `run_pipeline.py`.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col

spark = get_spark("novashop-m06")

# Tres where + select = receta más larga. Todavía NO hay viaje.
planned = (
    spark.read.parquet(str(STAGING / "fact_lines"))
    .where(col("is_billable"))
    .where(col("gmv_line") > 0)
    .where(col("channel_norm").isin("web", "app"))
    .select("order_id", "customer_id", "gmv_line", "order_month")
)

# Objeto (papel de la receta), NO el número de filas
print("¿qué es?", type(planned).__name__)
print(planned)


### Paso 2 — `explain`: ver qué haría **sin** hacerlo

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Copia y rellena:

```
Paso 2. explain imprime la receta. No recorre el Parquet. No da el count.
Job Id antes: ___   Job Id después: ___   (igual)
En el helper: fact_lines sí, is_billable sí, Filter sí.
En el tocho, Ctrl+F: una línea de Scan/fact_lines y una de Filter/PushedFilters.
Jobs: sin fila nueva. El árbol Scan+Filter AÚN no está en SQL: no hemos viajado.
```

**Cómo leer el tocho** (no de arriba abajo): Ctrl+F en la salida de la celda.

| Buscas | Quiere decir |
|--------|----------------|
| `fact_lines` / *FileScan* / *Scan parquet* | de aquí salen las filas |
| `is_billable` / `gmv_line` / `channel_norm` | tus `where` están en la receta |
| *PushedFilters* | el `where` se empujó al scan (no “leer todo y filtrar al final del todo”) |
| *Filter* | un `where` como paso |
| *Project* | el `select` de cuatro columnas |

El helper de la celda te imprime **sí/NO** para no perderte. El tocho es por si quieres copiar **una** línea de Scan y **una** de Filter.

**UI de este paso:**

1. Anota Job Id más alto.
2. Ejecuta. El notebook se llena de texto; **no** sale el entero del count.
3. Recarga **Jobs**: el más alto **sigue igual**. `explain` no es un viaje.
4. **SQL / DataFrame**: igual que antes (vacío, o las consultas *viejas*). Esta receta todavía no se ejecutó, así que **no** aparece un árbol nuevo.
5. **Storage**: vacío.

Esto es el aha: ya viste Scan y Filter **sin** haber leído el Parquet.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Helper: `sí` en `fact_lines`, `is_billable`, `Filter`. El tocho no termina en un entero. Job Id **igual**.

**Por qué este paso.** Si Job Id sube, no has hecho `explain`: has hecho `count`. Si el helper dice NO en `is_billable`, estás explain-ando otro DataFrame.


In [ ]:
def receta(df, etiqueta):
    # Versión corta del explain: sí/NO, sin leer el tocho como un libro
    txt = df._jdf.queryExecution().executedPlan().toString()
    print("=== Receta de", etiqueta, "(aún NO ha corrido) ===")
    for pal in (
        "fact_lines",
        "is_billable",
        "gmv_line",
        "channel_norm",
        "Filter",
        "Project",
    ):
        print("   ", "sí" if pal in txt else "NO", pal)

receta(planned, "planned")
print("----- tocho (Ctrl+F: Scan y Filter) -----")
planned.explain("formatted")


### Paso 3 — `count`: ahora sí viaja (un job, entonces hay qué pintar)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Copia y rellena:

```
Paso 3. count recorre el Parquet y aplica la receta que ya vimos en el explain.
Filas = ___   (varios cientos; cobrable + GMV>0 + web/app)
Job Id antes: ___   Job Id después: ___   (este SÍ sube: +1)
En Jobs: una fila nueva, Description con "count".
En SQL: una consulta nueva; al abrirla veo Scan y Filter (el explain, pero ejecutado).
En el DAG de Jobs (cajas grandes) NO busqué la palabra Filter; son stages.
Storage sigue vacío (no hay cache).
```

**UI de este paso (aquí por fin hay dibujo):**

1. Anota Job Id más alto (el de los pasos 1–2).
2. Ejecuta la celda. En el notebook: un **entero**.
3. Recarga **Jobs**. Hay **una fila nueva**. El más alto es el anterior **+1**. Description: `count` (el sitio del `NativeMethod…` da igual). Duration: irrelevante. Stages: `1/1` o `2/2` (el 2.º junta los recuentos parciales).
4. Clic en **ese** Job Id (el nuevo, no uno viejo).
   - “DAG Visualization”: cajas azules = **stages**, no cada operador. Nombres tipo *WholeStageCodegen*, *Exchange*, *count*. **No está mal que no ponga Filter.** Filter vive *dentro* de la etapa que lee.
   - Si hay **dos** cajas unidas: una leyó y contó trozos; la otra sumó el total. La flecha / *Exchange* es el shuffle de ese count, no un join tuyo.
5. Pestaña **SQL / DataFrame**. Una consulta nueva, misma duración grosera, asociada a ese job. Clic → Details.
   - Aquí sí: *FileScan parquet* / *Scan*, *Filter* / *PushedFilters*, *Project*, y **HashAggregate** (porque `count` suma).
   - Compáralo con el `explain` del paso 2: Scan y Filter son la misma receta. El Aggregate **extra** es el count (el explain de `planned` aún no sumaba: solo iba a producir filas).
6. **Storage**: sigue vacío.

Si cada celda te crea un job, tienes un `.show()` de debug: coméntalo.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Un entero **> 0**, varios cientos. Job Id **+1**. SQL muestra Scan y Filter. Storage vacío.

**Por qué este paso.** El entero no es la prueba. La prueba es la fila nueva en Jobs y que el árbol SQL cuadra con el explain del paso 2.


In [ ]:
# Viaje: abre fact_lines, aplica la receta, devuelve un entero
n = planned.count()
print("líneas cobrables web/app con GMV > 0 =", n)
print("mira Jobs: una fila nueva. SQL: árbol Scan+Filter+Aggregate")


## Prueba tú — Un `where` extra alarga la receta, no viaja

No copies y listo: **cambia** lo que indica el texto y mira si cuadra con **Qué tienes que ver**.

Sigue **sin** `count`. Copia:

```
Prueba. mas_estrecho = planned con gmv_line > 50.
print me dio otro DataFrame[…], no un entero.
Helper: planned no tiene por qué mencionar 50; mas_estrecho sí (gmv_line).
Job Id igual que al terminar el paso 3 (___). No hay fila nueva.
Si contara mas_estrecho, el n sería MENOR y nacería otro job: eso ya sería otro viaje.
```

**UI:** recarga Jobs **antes** y **después**. El más alto no cambia. SQL no añade consulta. El helper es el A/B: dos recetas, cero viajes.

**Qué tienes que ver.** Dos bloques `sí/NO`. `mas_estrecho` sigue mostrando `gmv_line`. Job Id **igual** que al final del paso 3.


In [ ]:
mas_estrecho = planned.where(col("gmv_line") > 50)
print("sigue siendo un objeto:", mas_estrecho)
receta(planned, "planned (3 where)")
receta(mas_estrecho, "mas_estrecho (4 where; GMV > 50)")
print("Job Id: el de después del count. Si sube, has lanzado un count sin querer.")


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

Cierra con **este** bloque en Markdown (rellenado; no otro ensayo):

```
Escribir where no recorre el Parquet; alarga la receta.
print enseña el objeto. explain enseña Scan/Filter sin viajar (Job Id ___ → ___ , igual).
count sí viaja: filas = ___ ; Job Id ___ → ___ (+1).
En Jobs el DAG son stages (no busqué Filter). En SQL vi Scan y Filter como en el explain.
Storage vacío en todo el lab.
Un where extra (gmv > 50) no creó job.
```


## Mejora — formatted vs el tocho extended (y dónde está en la UI)

`explain("formatted")` es el físico, el de clase. `explain(True)` suelta cuatro capas (parsed → analyzed → optimized → physical): más ruido. Quédate con **Physical Plan** / formatted.

`PushedFilters` junto al FileScan = el `where` se empujó al scan.

En la UI: SQL → tu `count` → Details. Ese árbol es el físico **después** de viajar. El `explain` del paso 2 era el mismo tipo de árbol **antes** de viajar.

Si te atasca, el código está en la celda siguiente.


In [ ]:
print("===== formatted (el de clase) =====")
planned.explain("formatted")
print("===== extended (busca Physical Plan y PushedFilters; ignora el resto) =====")
planned.explain(True)


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| UI vacía / 404 | 4040 no reenviado, o la sesión está en 4041 | Ports → globo 4040; o `spark.stop()` + `get_spark()` y prueba 4041 |
| Job Id sube en print/explain | `count`/`show` en la misma celda, o clicaste un job viejo y pensaste que era nuevo | Quita acciones; compara el número *antes/después*, no el dibujo viejo |
| En el DAG de Jobs no veo Filter | Normal: esas cajas son stages | Árbol con Filter: pestaña SQL, o el `explain` del notebook |
| SQL vacío después del count | No recargaste, o miras otra app (nombre ≠ novashop-m06) | F5; comprueba el nombre arriba |
| Storage con algo | Cache de otra celda / de M06-02 | `spark.catalog.clearCache()` |
| Cada celda crea un job | `show` de debug | Coméntalo mientras mides Jobs |
| Dos sesiones | `SparkSession()` a mano | Solo `get_spark()` |
| explain → AnalysisException | Columna mal escrita | Para planificar también resuelve nombres: corrige el typo |
| No está fact_lines | Saltaste M03 | `run_pipeline.py` o lab M03-02 |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M06-02 cache](03-lab-cache-particionado.ipynb).
